# K01 · The locked confirmatory test (run ONCE, after the freeze)

Runs the pre-registered confirmatory analysis on the confirmatory events, exactly as frozen in
`docs/preregistration.md`. **Run it once.** Whatever it prints is the result; a rerun is only allowed to recover
from a crash, and must be logged as such.

**It refuses to start** unless the plan is frozen (`src/damagesar_bench/freeze.py`): `docs/FREEZE.md` exists with
the freeze date, git tag, the plan's DOI and the test level; the tag is an ancestor of this commit;
`configs/events.yaml`'s `freeze_date` matches; today is on or after it; and the override variable is not set.

What it runs (leave-one-event-out over **all** labelled events, probes trained on every other event):
* **RQ1** DINOv3 ViT-B/16 vs ImageNet ViT-B/16, outline pooling, native resolution (primary).
* **RQ2** C2 matched pooling vs outline, DINOv3 (primary, **only if Gate 1 passed**: read from the g01 run).
* **RQ4** DINOv3 vs the image-free control; outline vs the 10 m ring (exploratory).
* **RQ3** C3 label-free prevalence on each confirmatory event (exploratory).
Comparisons use the confirmatory events only; the interval level comes from the freeze record (Bonferroni).

* **Before running:** fresh session, Runtime → Change runtime type → **T4 GPU**. Secrets: `GH_TOKEN`, `HF_TOKEN`.
  Needs the g01 run in Drive (for the Gate 1 verdict).
* **Time:** about 2 hours (about 5 compute units). **Outputs:** this run's folder.

In [ ]:
NOTEBOOK_NAME = "k01_locked_confirmatory_test"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# 0. Is the plan frozen? (stops here if not)
!git -C /content/damagesar-bench fetch -q --tags
from damagesar_bench.config import load_events
from damagesar_bench.freeze import check_freeze
events = load_events("/content/damagesar-bench/configs/events.yaml")
FREEZE = check_freeze("/content/damagesar-bench", events)
print("Plan frozen:", FREEZE)

In [ ]:
# GPU, Hugging Face token, environment
import os, json, torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as ex:
    raise RuntimeError("HF_TOKEN is missing: add it in Colab -> Secrets (key icon), switch on notebook access, "
                       "then Run all again.") from ex
from damagesar_bench.reproduce import environment
ENV = {**environment(), "commit": COMMIT}
print(json.dumps(ENV, indent=1))

In [ ]:
# 1. Data, downloaded fresh to /content (BRIGHT ~3.4 GB from Hugging Face; instance labels 31 MB from Zenodo)
import zipfile, urllib.request
from huggingface_hub import hf_hub_download
ROOT = "/content/bright"
os.makedirs(ROOT, exist_ok=True)
for name in ["post-event.zip", "target.zip"]:
    folder = os.path.join(ROOT, name.replace(".zip", ""))
    if not (os.path.isdir(folder) and len(os.listdir(folder)) > 4000):
        p = hf_hub_download("Kullervo/BRIGHT", name, repo_type="dataset", local_dir="/content/zips")
        with zipfile.ZipFile(p) as z:
            z.extractall(ROOT)
    print(name, "->", len(os.listdir(folder)), "files")
LABELS = "/content/zips/cvprw26_trainval_instance_labels.zip"
if not os.path.exists(LABELS):
    urllib.request.urlretrieve(
        "https://zenodo.org/records/19216381/files/cvprw26_trainval_instance_labels.zip?download=1", LABELS)
print("instance labels:", round(os.path.getsize(LABELS) / 1e6, 1), "MB")

In [ ]:
# 3. Building table (counts, exclusions, building size in tokens)
from damagesar_bench.config import load_bench
from damagesar_bench.data.buildings import build_building_table, summarise
cfg = load_bench("/content/damagesar-bench/configs/bench.yaml")
B, stats = build_building_table(LABELS, events, cfg.nodata, bright_root=ROOT, target_res_m=0.3)
BSUM = summarise(B)
BSUM.to_csv(f"{RUN}/buildings_summary.csv", index=False)
print(stats)
print(BSUM[["event", "role", "buildings", "kept", "pct_lt_4_tokens_native", "pct_lt_4_tokens_target"]].to_string(index=False))

In [ ]:
# 4. Gate 1 verdict (from g01) and the C2 shifts for every event (footprints + SAR only, no labels)
import glob, gc
import pandas as pd
G1 = sorted(glob.glob(f"{PROJECT}/04_outputs/runs/*_g01-gate1-c2-matched-pooling/gate1.json"))
assert G1, "missing the g01 run (Gate 1): run g01 before the locked test"
GATE1 = json.load(open(G1[-1]))
RUN_RQ2 = bool(GATE1["passed"])
print("Gate 1 passed:", RUN_RQ2, "-> RQ2", "runs" if RUN_RQ2 else "is not run")
from damagesar_bench.c2_offsets import matched_shifts, measure_offsets
from damagesar_bench.encoders.extract import read_bright_tile, write_features
labelled = [e for e, c in events.events.items() if c.units != "none"]
SHIFTS = None
if RUN_RQ2:
    off = measure_offsets(B, lambda t: read_bright_tile(ROOT, t), labelled, factor=4, max_shift_m=40, n_null=3,
                          seed=0, log=print)
    off.to_csv(f"{RUN}/offsets.csv", index=False)
    SHIFTS = matched_shifts(off)

In [ ]:
# 5. Features on every labelled event, native resolution
from damagesar_bench.controls.imagefree import image_free_features
from damagesar_bench.encoders import get_encoder
from damagesar_bench.experiments.resolution import extract_at_resolutions, runs_table
FEATS, timing, revisions = [], [], {}
FEATS.append(str(write_features(image_free_features(B, events), "/content/imagefree.parquet")))
plan = {"dinov3_vitb16": ["outline", "ring:10"] + (["matched"] if RUN_RQ2 else []), "imagenet_vitb16": ["outline"]}
for name, methods in plan.items():
    enc = get_encoder(name); revisions[name] = enc.weights_revision
    f, runs = extract_at_resolutions(B, lambda t: read_bright_tile(ROOT, t), enc, methods, [None],
                                     f"/content/features/{name}", events, log=print, shifts=SHIFTS)
    FEATS.append(str(write_features(f, f"/content/{name}.parquet"))); timing.append(runs_table(runs))
    del f, enc; gc.collect(); torch.cuda.empty_cache()
pd.concat(timing).to_csv(f"{RUN}/timing.csv", index=False)
json.dump(revisions, open(f"{RUN}/weights_revisions.json", "w"), indent=1)

In [ ]:
# 6. The locked run: probes on all other events; comparisons on the confirmatory events only
from IPython.display import Markdown
from damagesar_bench.run import ComparisonSpec, run_benchmark
D, I = "dinov3_vitb16@native", "imagenet_vitb16@native"
comps = [ComparisonSpec((D, "outline"), (I, "outline"), "RQ1 DINOv3 vs ImageNet (outline)")]
if RUN_RQ2:
    comps.append(ComparisonSpec((D, "matched"), (D, "outline"), "RQ2 C2 matched vs outline (DINOv3)"))
comps += [ComparisonSpec((D, "outline"), ("imagefree", "geometry"), "RQ4 DINOv3 vs image-free"),
          ComparisonSpec((D, "outline"), (D, "ring10m"), "RQ4 building vs surroundings (DINOv3)")]
out = run_benchmark(FEATS, comps, events, f"{RUN}/bench_run", n_boot=2000, seed=0, level=FREEZE.level,
                    event_set="confirmatory", only_compared=True)
display(Markdown(open(f"{out}/report.md", encoding="utf-8").read()))
c = pd.read_csv(f"{out}/comparisons.csv")
print(c[["label", "strength", "n_events", "estimate", "lower", "upper", "verdict", "sign_flip_p"]].round(4).to_string(index=False))

In [ ]:
# 7. RQ3: C3 label-free prevalence on each confirmatory event (exploratory; rule as frozen)
from damagesar_bench.recal.prevalence import estimate_all
S = pd.read_parquet(f"{out}/fold_scores.parquet")
est = estimate_all(S, D, "outline")
est = est[est["event"].isin(events.keys_with_role("confirmatory"))]
est.to_csv(f"{RUN}/c3_confirmatory.csv", index=False)
display(est[["event", "n", "true_prevalence", "baseline", "em", "bbse", "abstain", "estimate", "rel_error",
             "baseline_rel_error", "reason"]].round(3))

In [ ]:
os.makedirs(f"{PROJECT}/03_notebooks/locked", exist_ok=True)

# File the executed notebook (with outputs) into this run's folder and 03_notebooks/locked/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/locked/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))